---
title: "Exercise 7: scDRS Cell-Type Analysis"
subtitle: "Post-GWAS Analysis Course"
format:
  html:
    embed-resources: true
    toc: true
    toc-depth: 3
execute:
  message: false
  warning: false
jupyter: python
---

# Overview

This notebook uses scDRS to connect the MAGMA gene set to cell types in the Jerber dopaminergic neuron dataset and to explore how robust those associations are.

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- prepare the gene-set and single-cell inputs needed by scDRS
- run the scoring and group-level association steps
- interpret scDRS output in the context of cell-type enrichment and sensitivity analysis
:::

 ## Table of Contents

* [Set up](#section_1)     
* [Single cell RNA sequencing dataset](#section_2) 
* [Prepare the input files for scDRS](#section_3) 
    * [Covariance file](#section_3_1)
    * [Gene set file](#section_3_2)
* [Run scDRS](#section_4)     
* [Run group-level association](#section_5)

# 1. Set up <a class="anchor" id="section_1"></a>

In [1]:
#Load the different libraries 
import scdrs
import scanpy as sc
from anndata import AnnData
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

# 2. Single-cell RNA sequencing dataset: dopaminergic neuron differentiation <a class="anchor" id="section_2"></a>
For this example we are going to use a dataset that was build by differentiating "215 human induced pluripotent stem cell (iPSC) lines toward a midbrain neural fate, including dopaminergic neurons, and use single-cell RNA sequencing (scRNA-seq) to profile over 1 million cells across three differentiation time points." 
The original publication can be found here: https://pubmed.ncbi.nlm.nih.gov/33664506/ 
<br>
The dataset is divided into differentiating neurons cultivated for 11, 30, or 52 days

In [3]:
#Load the dataset in H5AD format
#We focus here on the 11day neurons

adata = sc.read_h5ad("./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad")


The scRNA seq dataset is organised in H5AD format, and can be read using the library AnnData (https://anndata.readthedocs.io/en/latest/). 
H5AD files have the following structure: 

<div>
<img src="figures/anndata_schema.svg" width = 500 /> 
</div>

In [4]:
# Explore the structure of the dataset and answer the following questions:
adata

AnnData object with n_obs × n_vars = 253381 × 32738
    obs: 'sample_index', 'sample_id', 'donor_id', 'cluster_id', 'celltype', 'time_point', 'pool_id', 'treatment'
    var: 'gene_ids-0', 'gene_ids-1', 'gene_ids-2', 'gene_ids-3', 'gene_ids-4', 'gene_ids-5', 'gene_ids-6', 'gene_ids-7', 'gene_ids-8', 'gene_ids-9', 'gene_ids-10', 'gene_ids-11', 'gene_ids-12', 'gene_ids-13', 'gene_ids-14', 'gene_ids-15', 'gene_ids-16', 'gene_ids-17', 'gene_ids-18', 'gene_ids-19', 'gene_ids-20', 'gene_ids-21', 'gene_ids-22', 'gene_ids-23', 'gene_ids-24', 'gene_ids-25', 'gene_ids-26', 'gene_ids-27', 'gene_ids-28', 'gene_ids-29', 'gene_ids-30', 'gene_ids-31', 'gene_ids-32', 'gene_ids-33', 'gene_ids-34', 'gene_ids-35', 'gene_ids-36', 'gene_ids-37'
    obsm: 'X_umap'

In [5]:
# Q1. How many cells are contained in the dataset?

# Q2. How many genes?

# Q3. What type of information contains 'obs'?

# Q4. What type of information contains 'var'? 

# Q5. What type of infomation contains 'obsm'?


In [6]:
# You can also view a snippet of the table using
adata.obs

,sample_index,sample_id,donor_id,cluster_id,celltype,time_point,pool_id,treatment
index,,,,,,,,
AAACCTGAGAACAACT-1-0,0,5245STDY7352549,HPSI0714i-iudw_1,2,P_FPP,D11,pool1,NONE
AAACCTGAGACAAGCC-1-0,0,5245STDY7352549,HPSI0614i-liqa_1,1,P_FPP,D11,pool1,NONE
AAACCTGAGACGCACA-1-0,0,5245STDY7352549,HPSI1113i-podx_1,1,P_FPP,D11,pool1,NONE
AAACCTGAGGAATGGA-1-0,0,5245STDY7352549,HPSI0114i-eipl_1,0,FPP,D11,pool1,NONE
AAACCTGCACGTCTCT-1-0,0,5245STDY7352549,HPSI0914i-suop_5,1,P_FPP,D11,pool1,NONE
...,...,...,...,...,...,...,...,...
TTTGGTTTCTTCAACT-1-37,37,5245STDY7962568,HPSI0813i-guss_1,0,FPP,D11,pool13,NONE
TTTGTCAAGCTGCAAG-1-37,37,5245STDY7962568,HPSI0813i-peoj_1,0,FPP,D11,pool13,NONE
TTTGTCAGTAACGCGA-1-37,37,5245STDY7962568,HPSI0714i-kute_5,0,FPP,D11,pool13,NONE


::: callout-note
**Question.** What format do gene names have in the dataset?
:::

# 3. Prepare the input files for scDRS <a class="anchor" id="section_3"></a>
scDRS requires 3 different files as input: the H5AD file containing the scRNA seq data, a covariate file associated to the H5AD file, and the gene set file.
A description of the different input and output file formats for scDRS can be found here: https://martinjzhang.github.io/scDRS/file_format.html  

## 3.1 Covariate file <a class="anchor" id="section_3_1"></a>
The covariate file (.cov) contains information about the covariate we would like to include in the analysis. The first column corresponds to the cell names (matching adata.obs_names), and the other columns contain the covariates (e.g. n_genes, sex_male, age...) for each cell. 
<br>
The covariates you decide to include should be adapted to the dataset you are using and how it was created.
<br>
Here, we will include the donor ID, the number of genes expressed by the cell, and a constant.

In [39]:
os.makedirs("./output/scdrs/", exist_ok=True)

In [40]:
#Initiate the covariate table
cov = adata

#Create a column with the number of genes:
sc.pp.filter_cells(cov, min_genes=0)

#New donor column:
cov.obs["donor_idv2"] = cov.obs["donor_id"].astype("category").cat.codes

#Add a column containing a constant value
cov.obs["const"] = 1

#Select the columns we need and save the file
file_cov = cov.obs[["const", "n_genes", "donor_idv2"]]
file_cov.to_csv("./output/scdrs/jeber_11day_neurons.cov", sep = "\t")

## 3.2 Gene set file <a class="anchor" id="section_3_2"></a>
The gene set file (.gs) contains the list of gene for each trait we want to consider. The first column contains the trait name, and the second column contains the geneset. The geneset corresponds to the gene id, and the MAGMA z-score or p-value for that gene. 

For this example, we want to build a .gs file with the top 100 genes identified by MAGMA. 

In [41]:
#Load the output files from MAGMA
magma_out = pd.read_fwf("./output/magma/adhd_full.genes.out")
magma_out.head()

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,ZSTAT,P
0,84069,1,891872,920488,7,1,225534,0.65538,0.25611
1,84808,1,900579,927473,20,3,225534,1.22680,0.10994
2,57801,1,924342,946608,29,3,225534,0.87823,0.18991
3,9636,1,938847,959920,15,1,225534,0.47696,0.31669
4,375790,1,945503,1001499,7,1,225534,0.63903,0.26140


In [42]:
#As we can see in the first few rows of the MAGMA output, the genes are designated by numbers.
#We use a file with the MAGMA gene numbers and corresponding gene symbols to fix that

#Load the reference file with gene number and corresponding gene symbol
ncbi_ref = pd.read_csv("./reference_data/NCBI37.3.gene.loc", sep = "\t", header = None)
ncbi_ref = ncbi_ref.iloc[:,[0,5]]
ncbi_ref.columns = ["GENE","GENE_symbol"]
ncbi_ref.head()

,GENE,GENE_symbol
0,79501,OR4F5
1,100996442,LOC100996442
2,729759,OR4F29
3,81399,OR4F16
4,148398,SAMD11


In [43]:
#Create a table with the gene name and the corresponding z-score

#Add the gene symbol column
df = magma_out.merge(ncbi_ref, how = 'inner', on = 'GENE')

#Select the columns we need for scDRS
df = df[['GENE_symbol', 'ZSTAT']]

#Rename the ZSTAT column with the name of the trait
df.columns = ["GENE", "ADHD"]
df.head()

#Save the file 
df.to_csv("./output/scdrs/zfile_adhd_magma.tsv", sep = "\t", index = False)

In [44]:
#Use scdrs munge-gs to create the .gs file
#We munge the file to keep the TOP 100 GENES

munge_command = "scdrs munge-gs \
--out-file ./output/scdrs/munge_adhd_magma_100.gs \
--zscore-file ./output/scdrs/zfile_adhd_magma.tsv \
--weight zscore \
--n-max 100"

os.system(munge_command)

******************************************************************************
* Single-cell disease relevance score (scDRS)
* Version 1.0.2
* Martin Jinye Zhang and Kangcheng Hou
* HSPH / Broad Institute / UCLA
* MIT License
******************************************************************************
Call: scdrs munge-gs \
--zscore-file ./output/scdrs/zfile_adhd_magma.tsv \
--weight zscore \
--n-min 100 \
--n-max 100 \
--out-file ./output/scdrs/munge_adhd_magma_100.gs

--zscore-file loaded: n_gene=18112, n_trait=1 (sys_time=0.0s)
Print info for the first 3 traits and first 10 genes
Traits               ['ADHD']
PLEKHN1              [0.65538]
PERM1                [1.2268]
HES4                 [0.87823]
ISG15                [0.47696]
AGRN                 [0.63903]
C1orf159             [0.44094]
TTLL10               [0.63568]
TNFRSF18             [-1.5323]
TNFRSF4              [-1.4947]
UBE2J2               [-0.31362]
--zscore-file have values above 1 or below 0. Seems fine.

Finish mu

0

::: callout-note
Open and inspect the munged gene set . **Question.** What does the output of the munging step look like? You can also directly check the file from the file browser in jupyterLab.
:::

## 4. Run scDRS on the 11day neurons dataset <a class="anchor" id="section_4"></a>
We now have prepared all the input files necessary to run scDRS. 
<br>
We can run the compute-score function, which will calculate a score for each cell in the dataset. This command takes as input the h5ad file, the covariate file, the gene set file, and additional arguments related to which score to return.

In [2]:
#This step took me more than 30 minutes

scdrs_command = "scdrs compute-score \
--h5ad-file ./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad \
--h5ad-species human \
--gs-file ./output/scdrs/munge_adhd_magma_100.gs \
--gs-species human \
--cov-file ./reference_data/scDRS_Jerber_dataset/11day_DA_neurons_wSex.tsv \
--flag-filter-data True \
--flag-raw-count True \
--flag-return-ctrl-raw-score False \
--flag-return-ctrl-norm-score True \
--out-folder ./output/scdrs/ "

print(f"The command to use is below. We do not run it as it takes some time, but we have already generated the output files.\n{scdrs_command}")

The command to use is below. We do not run it as it takes some time, but we have already generated the output files.
scdrs compute-score --h5ad-file ./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad --h5ad-species human --gs-file ./output/scdrs/munge_adhd_magma_100.gs --gs-species human --cov-file ./reference_data/scDRS_Jerber_dataset/11day_DA_neurons_wSex.tsv --flag-filter-data True --flag-raw-count True --flag-return-ctrl-raw-score False --flag-return-ctrl-norm-score True --out-folder ./output/scdrs/ 


::: callout-note
**Question.** Can you tell why the software outputs two files .score.gz and .full_score.gz? 

:::

## 5. Run the group level analysis <a class="anchor" id="section_5"></a>
Now that we have computed the disease score for each cell, we can compute group-level associations. Here, the groups that we will consider are the different cell types in the dataset.
<br>
We use the perform-downstream command, which will calculate an association p-value for each cell type in the dataset, based on the disease score for each cell, and the scores for control gene sets. The function takes as input the h5ad file, the covariate file, the scores calculated for each cell, and the name of the group-level we want to analyse.

In [3]:
#This took around 30min to run

group_level_command = "scdrs perform-downstream \
--h5ad-file ./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad \
--score-file ./output/scdrs/ADHD.full_score.gz \
--out-folder ./output/scdrs/ \
--group-analysis celltype \
--flag-filter-data True \
--flag-raw-count True"

print(f"The command to use is below. We do not run it as it takes some time, but we have already generated the output files.\n{group_level_command}")

The command to use is below. We do not run it as it takes some time, but we have already generated the output files.
scdrs perform-downstream --h5ad-file ./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad --score-file ./output/scdrs/ADHD.full_score.gz --out-folder ./output/scdrs/ --group-analysis celltype --flag-filter-data True --flag-raw-count True


::: callout-note
**Question.** How is the output file organised? You can use the file format page to understand the different columns.
:::